# Pipeline de Entrenamiento: Prediccion de Tarifas de Uber

En este notebook implementamos el pipeline completo de entrenamiento para predecir tarifas de Uber, incluyendo:

1. Carga y limpieza de datos
2. Ingenieria de features
3. Entrenamiento con MLflow tracking
4. Evaluacion del modelo
5. Comparacion Champion vs Challenger

## Dataset

Usamos el [Uber Fares Dataset](https://www.kaggle.com/datasets/yasserh/uber-fares-dataset) de Kaggle.

**Variables:**
- `fare_amount`: Tarifa del viaje en USD (variable objetivo)
- `pickup_datetime`: Fecha y hora de inicio del viaje
- `passenger_count`: Numero de pasajeros
- `pickup_longitude/latitude`: Coordenadas de recogida
- `dropoff_longitude/latitude`: Coordenadas de destino

In [ ]:
# Importaciones
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Carga de datos

Primero descarga el dataset de Kaggle y colocalo en `data/uber.csv`.

Link: https://www.kaggle.com/datasets/yasserh/uber-fares-dataset

In [ ]:
# Cargar datos crudos
df_raw = pd.read_csv("../data/uber.csv")
print(f"Dimensiones: {df_raw.shape}")
df_raw.head()

In [ ]:
# Informacion general
df_raw.info()

In [ ]:
# Estadisticas descriptivas
df_raw.describe().round(2)

In [ ]:
# Valores nulos
print("Valores nulos por columna:")
print(df_raw.isnull().sum())

## 2. Analisis exploratorio rapido

In [ ]:
# Distribucion de la tarifa (variable objetivo)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Todas las tarifas
axes[0].hist(df_raw["fare_amount"], bins=100, color="#1f77b4", edgecolor="k", alpha=0.7)
axes[0].set_title("Distribucion de Tarifas (todas)")
axes[0].set_xlabel("Tarifa (USD)")
axes[0].set_ylabel("Frecuencia")

# Filtrando outliers para mejor visualizacion
tarifas_razonables = df_raw[(df_raw["fare_amount"] > 0) & (df_raw["fare_amount"] < 100)]["fare_amount"]
axes[1].hist(tarifas_razonables, bins=80, color="#2ca02c", edgecolor="k", alpha=0.7)
axes[1].set_title("Distribucion de Tarifas ($0 - $100)")
axes[1].set_xlabel("Tarifa (USD)")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

print(f"Tarifa promedio: ${df_raw['fare_amount'].mean():.2f}")
print(f"Tarifa mediana: ${df_raw['fare_amount'].median():.2f}")

In [ ]:
# Distribucion de pasajeros
fig, ax = plt.subplots(figsize=(8, 5))
df_raw["passenger_count"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#ff7f0e", edgecolor="k")
ax.set_title("Distribucion de Numero de Pasajeros")
ax.set_xlabel("Pasajeros")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

## 3. Ingenieria de features

Transformamos los datos crudos en features utiles para el modelo:

- **Distancia**: Calculamos la distancia en km entre pickup y dropoff usando la formula de Haversine
- **Features temporales**: Extraemos hora, dia de la semana, mes
- **Features binarias**: Es fin de semana? Es hora pico?
- **Limpieza**: Eliminamos tarifas negativas, distancias imposibles, coordenadas invalidas

In [ ]:
# Aplicar ingenieria de features usando nuestro modulo
from src.data import feature_engineering

df = feature_engineering(df_raw)
df.head()

In [ ]:
# Ver las features resultantes
print("Features finales:")
for col in df.columns:
    print(f"  - {col}: {df[col].dtype}")
print(f"\nTotal: {df.shape[0]} filas, {df.shape[1]} columnas")

In [ ]:
# Visualizar la relacion distancia vs tarifa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
sample = df.sample(min(5000, len(df)), random_state=42)
axes[0].scatter(sample["distancia_km"], sample["fare_amount"], alpha=0.3, s=10)
axes[0].set_title("Tarifa vs Distancia")
axes[0].set_xlabel("Distancia (km)")
axes[0].set_ylabel("Tarifa (USD)")

# Tarifa promedio por hora
tarifa_por_hora = df.groupby("hora")["fare_amount"].mean()
axes[1].bar(tarifa_por_hora.index, tarifa_por_hora.values, color="#9467bd", edgecolor="k")
axes[1].set_title("Tarifa Promedio por Hora del Dia")
axes[1].set_xlabel("Hora")
axes[1].set_ylabel("Tarifa Promedio (USD)")

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlacion
fig, ax = plt.subplots(figsize=(10, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Matriz de Correlacion")
plt.tight_layout()
plt.show()

# Correlacion con la variable objetivo
print("\nCorrelacion con fare_amount:")
print(correlation_matrix["fare_amount"].drop("fare_amount").sort_values(ascending=False).to_string())

## 4. Entrenamiento con MLflow

Ahora entrenamos el modelo y registramos todo en MLflow a traves de DagsHub.

**Asegurate de tener tu archivo `.env` configurado con las credenciales de DagsHub.**

In [ ]:
# Configurar MLflow con DagsHub
from src.config import setup_mlflow
setup_mlflow()

In [ ]:
# Preparar datos para entrenamiento
from src.data import prepare_data

X_train, X_test, y_train, y_test = prepare_data(df)

### 4.1 Entrenar Random Forest

In [ ]:
from src.train import train_model
from src.evaluate import evaluate_model, log_metrics_to_mlflow, create_evaluation_plots

# Entrenar Random Forest
with mlflow.start_run(run_name="notebook-random-forest"):
    modelo_rf = train_model(X_train, y_train, model_type="random_forest")
    metricas_rf = evaluate_model(modelo_rf, X_test, y_test)
    log_metrics_to_mlflow(metricas_rf)
    
    y_pred_rf = modelo_rf.predict(X_test)
    create_evaluation_plots(y_test, y_pred_rf)
    
    mlflow.sklearn.log_model(modelo_rf, "model")
    rf_run_id = mlflow.active_run().info.run_id

print(f"\nRun ID: {rf_run_id}")
print(f"RMSE: {metricas_rf['rmse']:.4f}")
print(f"MAE:  {metricas_rf['mae']:.4f}")
print(f"R2:   {metricas_rf['r2']:.4f}")

### 4.2 Entrenar Gradient Boosting

In [ ]:
# Entrenar Gradient Boosting para comparar
with mlflow.start_run(run_name="notebook-gradient-boosting"):
    modelo_gb = train_model(X_train, y_train, model_type="gradient_boosting")
    metricas_gb = evaluate_model(modelo_gb, X_test, y_test)
    log_metrics_to_mlflow(metricas_gb)
    
    y_pred_gb = modelo_gb.predict(X_test)
    create_evaluation_plots(y_test, y_pred_gb)
    
    mlflow.sklearn.log_model(modelo_gb, "model")
    gb_run_id = mlflow.active_run().info.run_id

print(f"\nRun ID: {gb_run_id}")
print(f"RMSE: {metricas_gb['rmse']:.4f}")
print(f"MAE:  {metricas_gb['mae']:.4f}")
print(f"R2:   {metricas_gb['r2']:.4f}")

### 4.3 Comparacion de modelos

In [ ]:
# Comparar ambos modelos lado a lado
comparacion = pd.DataFrame({
    "Random Forest": metricas_rf,
    "Gradient Boosting": metricas_gb,
}).T

print("Comparacion de modelos:")
print("=" * 50)
print(comparacion.round(4).to_string())

# Determinar el mejor modelo
mejor = "Random Forest" if metricas_rf["rmse"] < metricas_gb["rmse"] else "Gradient Boosting"
print(f"\nMejor modelo: {mejor}")

In [ ]:
# Visualizar comparacion de predicciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest
sample_idx = np.random.choice(len(y_test), min(500, len(y_test)), replace=False)
y_test_sample = y_test.iloc[sample_idx]

axes[0].scatter(y_test_sample, y_pred_rf[sample_idx], alpha=0.4, s=15)
axes[0].plot([y_test_sample.min(), y_test_sample.max()], 
             [y_test_sample.min(), y_test_sample.max()], "r--", lw=2)
axes[0].set_title(f"Random Forest (RMSE={metricas_rf['rmse']:.2f})")
axes[0].set_xlabel("Tarifa Real (USD)")
axes[0].set_ylabel("Tarifa Predicha (USD)")

# Gradient Boosting
axes[1].scatter(y_test_sample, y_pred_gb[sample_idx], alpha=0.4, s=15, color="orange")
axes[1].plot([y_test_sample.min(), y_test_sample.max()], 
             [y_test_sample.min(), y_test_sample.max()], "r--", lw=2)
axes[1].set_title(f"Gradient Boosting (RMSE={metricas_gb['rmse']:.2f})")
axes[1].set_xlabel("Tarifa Real (USD)")
axes[1].set_ylabel("Tarifa Predicha (USD)")

plt.tight_layout()
plt.show()

## 5. Champion / Challenger

Usamos la logica de Champion/Challenger para registrar el mejor modelo en el Model Registry de MLflow.

In [ ]:
from src.champion_challenger import get_champion_metrics, compare_models, promote_to_champion, generate_report

# Usar las metricas del mejor modelo como Challenger
if metricas_rf["rmse"] < metricas_gb["rmse"]:
    challenger_metrics = metricas_rf
    challenger_run_id = rf_run_id
    print("Challenger: Random Forest")
else:
    challenger_metrics = metricas_gb
    challenger_run_id = gb_run_id
    print("Challenger: Gradient Boosting")

# Obtener metricas del Champion actual (si existe)
champion_metrics = get_champion_metrics()

# Comparar
comparison = compare_models(challenger_metrics, champion_metrics)
print(f"\nResultado: {comparison['reason']}")

In [ ]:
# Generar reporte
report = generate_report(comparison, challenger_metrics, champion_metrics)
print(report)

In [ ]:
# Promover a Champion si es mejor
# DESCOMENTA la siguiente linea para promover el modelo
# promote_to_champion(challenger_run_id)

## 6. Importancia de features

In [ ]:
# Importancia de features del mejor modelo
mejor_modelo = modelo_rf if metricas_rf["rmse"] < metricas_gb["rmse"] else modelo_gb
feature_names = X_train.columns
importances = mejor_modelo.feature_importances_

# Ordenar por importancia
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(indices)), importances[indices], color="#2ca02c", edgecolor="k")
ax.set_yticks(range(len(indices)))
ax.set_yticklabels([feature_names[i] for i in indices])
ax.invert_yaxis()
ax.set_title("Importancia de Features")
ax.set_xlabel("Importancia")
plt.tight_layout()
plt.show()

print("\nRanking de features:")
for i, idx in enumerate(indices, 1):
    print(f"  {i}. {feature_names[idx]}: {importances[idx]:.4f}")

## Conclusiones

- La **distancia** es la feature mas importante para predecir la tarifa (como era de esperarse)
- Las features temporales (hora, dia de la semana) aportan informacion adicional
- La hora pico y el fin de semana capturan patrones de demanda
- El numero de pasajeros tiene poca influencia en la tarifa

### Siguientes pasos

1. Experimentar con diferentes hiperparametros
2. Crear una rama, modificar parametros en `src/train.py` y abrir un PR
3. El GitHub Action entrenara automaticamente y publicara un reporte CML
4. Usar `/train` en Claude Code para entrenar interactivamente